# 🌈 VisionMate AI
## Image Captioning + Object Detection in One Website

**Goal:** Build one colorful Gradio website that can:
1. 📝 Generate a caption describing an uploaded image.
2. 🎯 Detect objects in the same image and draw bounding boxes around them.

**Models from Hugging Face**
- Image captioning: `Salesforce/blip-image-captioning-base`
- Object detection: `facebook/detr-resnet-50`

**Recommended:** Run this notebook in Google Colab. A GPU makes inference faster. The models can also run on CPU, but may be slower.

> Note: These models identify visual content; predictions may not always be correct. The first run downloads model files, so it needs an internet connection.

## Cell 1 — Install the required libraries

In [ ]:
!pip install -q -U transformers gradio torch pillow

## Cell 2 — Import libraries and check the device

In [ ]:
import torch
import gradio as gr
from transformers import pipeline
from PIL import Image, ImageDraw, ImageFont
import textwrap

device = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available())
print("Device:", "GPU" if device == 0 else "CPU")

## Cell 3 — Load both Hugging Face models

This may take a few minutes the first time because model files must be downloaded.

In [ ]:
print("Loading image captioning model...")
captioner = pipeline(
    task="image-to-text",
    model="Salesforce/blip-image-captioning-base",
    device=device
)

print("Loading object detection model...")
detector = pipeline(
    task="object-detection",
    model="facebook/detr-resnet-50",
    device=device
)

print("Both models loaded successfully!")

## Cell 4 — Create the image captioning function

In [ ]:
def generate_caption(image):
    if image is None:
        return "Please upload an image first."

    results = captioner(image.convert("RGB"), max_new_tokens=50)
    caption = results[0]["generated_text"]
    return caption

## Cell 5 — Create the object detection function

The function predicts object labels and confidence scores, then draws a box around each detected object.

In [ ]:
def detect_objects(image, threshold=0.50):
    if image is None:
        return None, "Please upload an image first."

    original = image.convert("RGB")
    results = detector(original)

    # Draw boxes on a copy so the original image remains unchanged.
    annotated = original.copy()
    draw = ImageDraw.Draw(annotated)
    detected_lines = []

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
    except OSError:
        font = ImageFont.load_default()

    for item in results:
        score = float(item["score"])
        if score < threshold:
            continue

        label = item["label"]
        box = item["box"]
        x1 = int(box["xmin"])
        y1 = int(box["ymin"])
        x2 = int(box["xmax"])
        y2 = int(box["ymax"])

        # Use a bright color for each bounding box.
        color = (255, 70, 120)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)

        text = f"{label} {score:.0%}"
        bbox = draw.textbbox((x1, max(0, y1 - 24)), text, font=font)
        draw.rectangle(bbox, fill=color)
        draw.text((x1, max(0, y1 - 24)), text, fill="white", font=font)

        detected_lines.append(f"• {label} — confidence {score:.1%}")

    if detected_lines:
        details = "Detected objects:\n" + "\n".join(detected_lines)
    else:
        details = "No objects found above this confidence threshold. Try lowering the threshold."

    return annotated, details

## Cell 6 — Test the models with an image

Upload an image in the cell below to test the functions before creating the website.

In [ ]:
# Optional quick test: upload an image using the notebook file picker.
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename = next(iter(uploaded))
    test_image = Image.open(filename).convert("RGB")
    display(test_image)
    print("Caption:", generate_caption(test_image))
    test_result, test_details = detect_objects(test_image)
    display(test_result)
    print(test_details)

## Cell 7 — Build the colorful website

Upload one image, then use either button:
- **Generate Caption** to describe the image.
- **Detect Objects** to show bounding boxes and confidence scores.

Both features are included in the same website.

In [ ]:
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #f5f3ff 0%, #eff6ff 48%, #fdf2f8 100%);
}
#main-title {
    text-align: center;
    background: linear-gradient(90deg, #7c3aed, #2563eb, #db2777);
    color: white;
    padding: 22px;
    border-radius: 18px;
    margin-bottom: 12px;
}
.feature-card {
    border: 1px solid #ddd6fe !important;
    border-radius: 16px !important;
    box-shadow: 0 6px 18px rgba(76, 29, 149, 0.08);
}
"""

def caption_for_ui(image):
    if image is None:
        return "⚠️ Please upload an image first."
    try:
        return "📝 " + generate_caption(image)
    except Exception as error:
        return f"Captioning error: {error}"

def detection_for_ui(image, threshold):
    if image is None:
        return None, "⚠️ Please upload an image first."
    try:
        annotated, details = detect_objects(image, threshold)
        return annotated, details
    except Exception as error:
        return image, f"Detection error: {error}"

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="pink"),
    css=custom_css,
    title="VisionMate AI"
) as demo:
    gr.Markdown(
        "# 🌈 VisionMate AI\n### Image Captioning + Object Detection in One Place",
        elem_id="main-title"
    )
    gr.Markdown(
        "Upload an image once, then generate a caption or detect the objects inside it."
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(
                type="pil",
                label="📤 Upload your image",
                height=320
            )
            threshold = gr.Slider(
                minimum=0.10,
                maximum=0.95,
                value=0.50,
                step=0.05,
                label="🎯 Detection confidence threshold"
            )
            with gr.Row():
                caption_button = gr.Button("✨ Generate Caption", variant="primary")
                detect_button = gr.Button("🔍 Detect Objects", variant="secondary")

        with gr.Column(scale=1):
            caption_output = gr.Textbox(
                label="📝 Image Caption",
                lines=4,
                placeholder="Your image caption will appear here..."
                )
            detected_image = gr.Image(
                type="pil",
                label="🎯 Detected Objects (with boxes)",
                height=320
            )
            detection_output = gr.Textbox(
                label="📋 Detection Details",
                lines=6,
                placeholder="Detected labels and confidence scores will appear here..."
            )

    caption_button.click(
        fn=caption_for_ui,
        inputs=input_image,
        outputs=caption_output
    )
    detect_button.click(
        fn=detection_for_ui,
        inputs=[input_image, threshold],
        outputs=[detected_image, detection_output]
    )

    gr.Markdown(
        "💡 **Tip:** Use a clear image with visible objects. "
        "Lower the confidence threshold to show more possible detections."
    )

demo.launch(share=True)

## Cell 8 — Stop the website (optional)

Run this only when you want to stop the Gradio app.

In [ ]:
demo.close()
print("VisionMate AI website stopped.")